# 01 — Preparação de Dados: Inventário e Dicionário de Dados (`KAN-10`)

- **Projeto:** Spaceship Titanic — Análise Estatística (Pós-Graduação em Deep Learning - CIn/UFPE)
- **Ticket Jira:** [KAN-10 — Criar inventário e dicionário de dados](https://megamatic.atlassian.net/browse/KAN-10)
- **Fonte dos Dados:** [Kaggle — Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic)
- **Responsável Principal:** Gustavo
- **Revisor Dedicado:** Helton
- **Documentação Complementar:** [`docs/dicionario_dados.md`](../docs/dicionario_dados.md) e [`docs/inventario_dados.md`](../docs/inventario_dados.md)

---

## 1. Objetivo da Etapa

Este notebook realiza a inspeção estrutural exaustiva das **14 variáveis originais** do conjunto de dados **Spaceship Titanic**, consolidando:
1. **Inventário físico e estrutural** dos ativos brutos (`train.csv`, `test.csv`, `sample_submission.csv`) em `data/raw/`.
2. **Classificação segundo a tipologia estatística** padronizada pela equipe:
   - **Categórica Nominal:** `HomePlanet`, `Destination`, `CryoSleep`, `VIP`, `Transported`
   - **Categórica Estruturada:** `PassengerId`, `Cabin`, `Name`
   - **Quantitativa Discreta:** `Age`
   - **Quantitativa Contínua:** `RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`
3. **Tabela formal do Dicionário de Dados** contendo: *Nome da Coluna*, *Tipo do Dado*, *Tipologia Estatística*, *Escala de Medida*, *Descrição de Negócio* e *Valores Possíveis / Amplitude*.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

RAW_DIR = Path("../data/raw")
train_path = RAW_DIR / "train.csv"
test_path = RAW_DIR / "test.csv"
sub_path = RAW_DIR / "sample_submission.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df_sub = pd.read_csv(sub_path)

print(f"train.csv carregado com sucesso: {df_train.shape[0]:,} linhas x {df_train.shape[1]} colunas")
print(f"test.csv carregado com sucesso:  {df_test.shape[0]:,} linhas x {df_test.shape[1]} colunas")
print(f"sample_submission.csv carregado: {df_sub.shape[0]:,} linhas x {df_sub.shape[1]} colunas")


train.csv carregado com sucesso: 8,693 linhas x 14 colunas
test.csv carregado com sucesso:  4,277 linhas x 13 colunas
sample_submission.csv carregado: 4,277 linhas x 2 colunas


---
## 2. Inventário dos Arquivos Brutos (`data/raw/`)

Inspeção de volumetria, granularidade, integridade de chaves (`PassengerId`), duplicatas e proporção global de células ausentes nos arquivos originais da competição.


In [2]:
inventario_arquivos = pd.DataFrame([
    {
        "Arquivo": "train.csv",
        "Caminho": "data/raw/train.csv",
        "Tamanho (Bytes)": train_path.stat().st_size,
        "Linhas (Registros)": df_train.shape[0],
        "Colunas": df_train.shape[1],
        "IDs Únicos (PassengerId)": df_train["PassengerId"].nunique(),
        "Linhas Duplicadas": int(df_train.duplicated().sum()),
        "Células Ausentes (NaN)": int(df_train.isna().sum().sum()),
        "% Células Ausentes": round(df_train.isna().sum().sum() / df_train.size * 100, 2),
        "Contém Alvo (Transported)": "Sim"
    },
    {
        "Arquivo": "test.csv",
        "Caminho": "data/raw/test.csv",
        "Tamanho (Bytes)": test_path.stat().st_size,
        "Linhas (Registros)": df_test.shape[0],
        "Colunas": df_test.shape[1],
        "IDs Únicos (PassengerId)": df_test["PassengerId"].nunique(),
        "Linhas Duplicadas": int(df_test.duplicated().sum()),
        "Células Ausentes (NaN)": int(df_test.isna().sum().sum()),
        "% Células Ausentes": round(df_test.isna().sum().sum() / df_test.size * 100, 2),
        "Contém Alvo (Transported)": "Não"
    },
    {
        "Arquivo": "sample_submission.csv",
        "Caminho": "data/raw/sample_submission.csv",
        "Tamanho (Bytes)": sub_path.stat().st_size,
        "Linhas (Registros)": df_sub.shape[0],
        "Colunas": df_sub.shape[1],
        "IDs Únicos (PassengerId)": df_sub["PassengerId"].nunique(),
        "Linhas Duplicadas": int(df_sub.duplicated().sum()),
        "Células Ausentes (NaN)": int(df_sub.isna().sum().sum()),
        "% Células Ausentes": round(df_sub.isna().sum().sum() / df_sub.size * 100, 2),
        "Contém Alvo (Transported)": "Template"
    }
])

display(inventario_arquivos)


,Arquivo,Caminho,Tamanho (Bytes),Linhas (Registros),Colunas,IDs Únicos (PassengerId),Linhas Duplicadas,Células Ausentes (NaN),% Células Ausentes,Contém Alvo (Transported)
0,train.csv,data/raw/train.csv,805421,8693,14,8693,0,2324,1.91,Sim
1,test.csv,data/raw/test.csv,372487,4277,13,4277,0,1117,2.01,Não
2,sample_submission.csv,data/raw/sample_submission.csv,59902,4277,2,4277,0,0,0.00,Template


### 2.1. Amostra Inicial dos Registros (`df_train.head()`)

In [3]:
display(df_train.head(10))

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
5,0005_01,Earth,False,F/0/P,PSO J318.5-22,44.0,False,0.0,483.0,0.0,291.0,0.0,Sandie Hinetthews,True
6,0006_01,Earth,False,F/2/S,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,Billex Jacostaffey,True
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,True
8,0007_01,Earth,False,F/3/S,TRAPPIST-1e,35.0,False,0.0,785.0,17.0,216.0,0.0,Andona Beston,True
9,0008_01,Europa,True,B/1/P,55 Cancri e,14.0,False,0.0,0.0,0.0,0.0,0.0,Erraiam Flatic,True


### 2.2. Inventário Técnico de Completude e Cardinalidade por Coluna (`train.csv` e `test.csv`)

In [4]:
inventario_colunas = []
for idx, col in enumerate(df_train.columns, start=1):
    n_train = len(df_train)
    null_tr = int(df_train[col].isna().sum())
    uniq_tr = int(df_train[col].nunique(dropna=True))
    
    if col in df_test.columns:
        null_te = int(df_test[col].isna().sum())
        pct_te = round(null_te / len(df_test) * 100, 2)
        uniq_te = int(df_test[col].nunique(dropna=True))
    else:
        null_te, pct_te, uniq_te = None, None, None

    inventario_colunas.append({
        "Ordem": idx,
        "Nome da Coluna": col,
        "Dtype (pandas)": str(df_train[col].dtype),
        "Não-Nulos (train)": n_train - null_tr,
        "Ausentes (train)": null_tr,
        "% Ausentes (train)": round(null_tr / n_train * 100, 2),
        "Valores Únicos (train)": uniq_tr,
        "Ausentes (test)": null_te,
        "% Ausentes (test)": pct_te,
        "Valores Únicos (test)": uniq_te,
    })

df_inventario_colunas = pd.DataFrame(inventario_colunas)
display(df_inventario_colunas)


,Ordem,Nome da Coluna,Dtype (pandas),Não-Nulos (train),Ausentes (train),% Ausentes (train),Valores Únicos (train),Ausentes (test),% Ausentes (test),Valores Únicos (test)
0,1,PassengerId,str,8693,0,0.00,8693,0.0,0.00,4277.0
1,2,HomePlanet,str,8492,201,2.31,3,87.0,2.03,3.0
2,3,CryoSleep,object,8476,217,2.50,2,93.0,2.17,2.0
3,4,Cabin,str,8494,199,2.29,6560,100.0,2.34,3265.0
4,5,Destination,str,8511,182,2.09,3,92.0,2.15,3.0
5,6,Age,float64,8514,179,2.06,80,91.0,2.13,79.0
6,7,VIP,object,8490,203,2.34,2,93.0,2.17,2.0
7,8,RoomService,float64,8512,181,2.08,1273,82.0,1.92,842.0
8,9,FoodCourt,float64,8510,183,2.11,1507,106.0,2.48,902.0
9,10,ShoppingMall,float64,8485,208,2.39,1115,98.0,2.29,715.0


---
## 3. Dicionário de Dados Formal e Tipologia Estatística (`KAN-10`)

Abaixo construímos programaticamente a tabela formal do **Dicionário de Dados** para as 14 variáveis originais do dataset **Spaceship Titanic**, contemplando:
- **Nome da Coluna**
- **Tipo do Dado**
- **Tipologia Estatística** (*Categórica Nominal*, *Categórica Estruturada*, *Quantitativa Discreta*, *Quantitativa Contínua*)
- **Escala de Medida** (*Nominal*, *Razão*)
- **Descrição de Negócio**
- **Valores Possíveis / Amplitude** (calculados diretamente sobre `train.csv`)


In [5]:
# Extração dinâmica de amplitudes e domínios reais em train.csv
cabin_parts = df_train["Cabin"].dropna().str.split("/", expand=True)
decks_unicos = sorted(cabin_parts[0].unique().tolist())
cabin_num_min, cabin_num_max = int(cabin_parts[1].astype(int).min()), int(cabin_parts[1].astype(int).max())
sides_unicos = sorted(cabin_parts[2].unique().tolist())

pid_parts = df_train["PassengerId"].str.split("_", expand=True)
grupo_min, grupo_max = pid_parts[0].min(), pid_parts[0].max()
n_grupos = pid_parts[0].nunique()
pp_min, pp_max = pid_parts[1].min(), pid_parts[1].max()

dicionario_dados = pd.DataFrame([
    {
        "Nome da Coluna": "PassengerId",
        "Tipo do Dado": "Texto (string)",
        "Tipologia Estatística": "Categórica Estruturada",
        "Escala de Medida": "Nominal (Identificador Composto)",
        "Descrição de Negócio": "Identificador único de cada passageiro no formato gggg_pp, onde gggg indica o grupo de viagem e pp é o índice sequencial do membro dentro do grupo.",
        "Valores Possíveis / Amplitude": f"Formato gggg_pp ({df_train['PassengerId'].min()} a {df_train['PassengerId'].max()}). {n_grupos} grupos distintos ({grupo_min}–{grupo_max}); índice pp de {pp_min} a {pp_max}."
    },
    {
        "Nome da Coluna": "HomePlanet",
        "Tipo do Dado": "Texto (string)",
        "Tipologia Estatística": "Categórica Nominal",
        "Escala de Medida": "Nominal",
        "Descrição de Negócio": "Planeta de residência permanente / embarque de onde o passageiro partiu.",
        "Valores Possíveis / Amplitude": "Categorias: {'Earth', 'Europa', 'Mars'} (+ valores ausentes NaN)."
    },
    {
        "Nome da Coluna": "CryoSleep",
        "Tipo do Dado": "Booleano (bool / object)",
        "Tipologia Estatística": "Categórica Nominal",
        "Escala de Medida": "Nominal (Binária / Dicotômica)",
        "Descrição de Negócio": "Indica se o passageiro optou por permanecer em animação suspensa (criossono) durante a viagem, ficando confinado à cabine sem realizar gastos a bordo.",
        "Valores Possíveis / Amplitude": "Valores booleanos: {False, True} (+ valores ausentes NaN)."
    },
    {
        "Nome da Coluna": "Cabin",
        "Tipo do Dado": "Texto (string)",
        "Tipologia Estatística": "Categórica Estruturada",
        "Escala de Medida": "Nominal (Composta: deck/num/side)",
        "Descrição de Negócio": "Código da cabine onde o passageiro está hospedado no formato deck/num/side (convés, número da cabine e lado: P = Port/Bombordo, S = Starboard/Boreste).",
        "Valores Possíveis / Amplitude": f"{df_train['Cabin'].nunique()} cabines únicas. deck em {decks_unicos}; num em [{cabin_num_min}, {cabin_num_max}]; side em {sides_unicos}."
    },
    {
        "Nome da Coluna": "Destination",
        "Tipo do Dado": "Texto (string)",
        "Tipologia Estatística": "Categórica Nominal",
        "Escala de Medida": "Nominal",
        "Descrição de Negócio": "Exoplaneta de destino onde o passageiro desembarcará ao final da viagem espacial.",
        "Valores Possíveis / Amplitude": "Categorias: {'TRAPPIST-1e', '55 Cancri e', 'PSO J318.5-22'} (+ valores ausentes NaN)."
    },
    {
        "Nome da Coluna": "Age",
        "Tipo do Dado": "Numérico Inteiro (float64 com NaN)",
        "Tipologia Estatística": "Quantitativa Discreta",
        "Escala de Medida": "Razão (Discreta em anos completos)",
        "Descrição de Negócio": "Idade do passageiro em anos completos no momento da viagem (0 representa bebês menores de 1 ano).",
        "Valores Possíveis / Amplitude": f"Inteiros em [{int(df_train['Age'].min())}, {int(df_train['Age'].max())}] anos (Média: {df_train['Age'].mean():.2f}, Mediana: {df_train['Age'].median():.1f}, DP: {df_train['Age'].std():.2f})."
    },
    {
        "Nome da Coluna": "VIP",
        "Tipo do Dado": "Booleano (bool / object)",
        "Tipologia Estatística": "Categórica Nominal",
        "Escala de Medida": "Nominal (Binária / Dicotômica)",
        "Descrição de Negócio": "Indica se o passageiro pagou pelo serviço especial VIP de alta prioridade a bordo da espaçonave.",
        "Valores Possíveis / Amplitude": "Valores booleanos: {False, True} (+ valores ausentes NaN)."
    },
    {
        "Nome da Coluna": "RoomService",
        "Tipo do Dado": "Numérico Real (float64)",
        "Tipologia Estatística": "Quantitativa Contínua",
        "Escala de Medida": "Razão",
        "Descrição de Negócio": "Valor monetário total faturado pelo passageiro em pedidos de serviço de quarto (Room Service).",
        "Valores Possíveis / Amplitude": f"Reais não-negativos em [{df_train['RoomService'].min():.1f}, {df_train['RoomService'].max():,.1f}] (Média: {df_train['RoomService'].mean():.2f}, Mediana: {df_train['RoomService'].median():.1f})."
    },
    {
        "Nome da Coluna": "FoodCourt",
        "Tipo do Dado": "Numérico Real (float64)",
        "Tipologia Estatística": "Quantitativa Contínua",
        "Escala de Medida": "Razão",
        "Descrição de Negócio": "Valor monetário total faturado pelo passageiro na praça de alimentação (Food Court).",
        "Valores Possíveis / Amplitude": f"Reais não-negativos em [{df_train['FoodCourt'].min():.1f}, {df_train['FoodCourt'].max():,.1f}] (Média: {df_train['FoodCourt'].mean():.2f}, Mediana: {df_train['FoodCourt'].median():.1f})."
    },
    {
        "Nome da Coluna": "ShoppingMall",
        "Tipo do Dado": "Numérico Real (float64)",
        "Tipologia Estatística": "Quantitativa Contínua",
        "Escala de Medida": "Razão",
        "Descrição de Negócio": "Valor monetário total faturado pelo passageiro no centro comercial (Shopping Mall).",
        "Valores Possíveis / Amplitude": f"Reais não-negativos em [{df_train['ShoppingMall'].min():.1f}, {df_train['ShoppingMall'].max():,.1f}] (Média: {df_train['ShoppingMall'].mean():.2f}, Mediana: {df_train['ShoppingMall'].median():.1f})."
    },
    {
        "Nome da Coluna": "Spa",
        "Tipo do Dado": "Numérico Real (float64)",
        "Tipologia Estatística": "Quantitativa Contínua",
        "Escala de Medida": "Razão",
        "Descrição de Negócio": "Valor monetário total faturado pelo passageiro no centro de relaxamento e bem-estar (Spa).",
        "Valores Possíveis / Amplitude": f"Reais não-negativos em [{df_train['Spa'].min():.1f}, {df_train['Spa'].max():,.1f}] (Média: {df_train['Spa'].mean():.2f}, Mediana: {df_train['Spa'].median():.1f})."
    },
    {
        "Nome da Coluna": "VRDeck",
        "Tipo do Dado": "Numérico Real (float64)",
        "Tipologia Estatística": "Quantitativa Contínua",
        "Escala de Medida": "Razão",
        "Descrição de Negócio": "Valor monetário total faturado pelo passageiro no convés de simulação de realidade virtual (VRDeck).",
        "Valores Possíveis / Amplitude": f"Reais não-negativos em [{df_train['VRDeck'].min():.1f}, {df_train['VRDeck'].max():,.1f}] (Média: {df_train['VRDeck'].mean():.2f}, Mediana: {df_train['VRDeck'].median():.1f})."
    },
    {
        "Nome da Coluna": "Name",
        "Tipo do Dado": "Texto (string)",
        "Tipologia Estatística": "Categórica Estruturada",
        "Escala de Medida": "Nominal (Texto Composto: Nome + Sobrenome)",
        "Descrição de Negócio": "Nome completo do passageiro (FirstName LastName), permitindo identificar laços familiares por sobrenome.",
        "Valores Possíveis / Amplitude": f"{df_train['Name'].nunique()} nomes únicos em train.csv (2.217 sobrenomes distintos) + valores ausentes NaN."
    },
    {
        "Nome da Coluna": "Transported",
        "Tipo do Dado": "Booleano (bool)",
        "Tipologia Estatística": "Categórica Nominal",
        "Escala de Medida": "Nominal (Binária / Alvo)",
        "Descrição de Negócio": "Variável resposta (Target): indica se o passageiro foi teletransportado para outra dimensão após a colisão com a anomalia espaço-temporal.",
        "Valores Possíveis / Amplitude": "Valores booleanos sem nulos: {True (50,36%), False (49,64%)}."
    }
])

display(dicionario_dados)


,Nome da Coluna,Tipo do Dado,Tipologia Estatística,Escala de Medida,Descrição de Negócio,Valores Possíveis / Amplitude
0,PassengerId,Texto (string),Categórica Estruturada,Nominal (Identificador Composto),"Identificador único de cada passageiro no formato gggg_pp, onde gggg indica o grupo de viagem e pp é o índice sequencial do membro dentro do grupo.",Formato gggg_pp (0001_01 a 9280_02). 6217 grupos distintos (0001–9280); índice pp de 01 a 08.
1,HomePlanet,Texto (string),Categórica Nominal,Nominal,Planeta de residência permanente / embarque de onde o passageiro partiu.,"Categorias: {'Earth', 'Europa', 'Mars'} (+ valores ausentes NaN)."
2,CryoSleep,Booleano (bool / object),Categórica Nominal,Nominal (Binária / Dicotômica),"Indica se o passageiro optou por permanecer em animação suspensa (criossono) durante a viagem, ficando confinado à cabine sem realizar gastos a bordo.","Valores booleanos: {False, True} (+ valores ausentes NaN)."
3,Cabin,Texto (string),Categórica Estruturada,Nominal (Composta: deck/num/side),"Código da cabine onde o passageiro está hospedado no formato deck/num/side (convés, número da cabine e lado: P = Port/Bombordo, S = Starboard/Boreste).","6560 cabines únicas. deck em ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'T']; num em [0, 1894]; side em ['P', 'S']."
4,Destination,Texto (string),Categórica Nominal,Nominal,Exoplaneta de destino onde o passageiro desembarcará ao final da viagem espacial.,"Categorias: {'TRAPPIST-1e', '55 Cancri e', 'PSO J318.5-22'} (+ valores ausentes NaN)."
5,Age,Numérico Inteiro (float64 com NaN),Quantitativa Discreta,Razão (Discreta em anos completos),Idade do passageiro em anos completos no momento da viagem (0 representa bebês menores de 1 ano).,"Inteiros em [0, 79] anos (Média: 28.83, Mediana: 27.0, DP: 14.49)."
6,VIP,Booleano (bool / object),Categórica Nominal,Nominal (Binária / Dicotômica),Indica se o passageiro pagou pelo serviço especial VIP de alta prioridade a bordo da espaçonave.,"Valores booleanos: {False, True} (+ valores ausentes NaN)."
7,RoomService,Numérico Real (float64),Quantitativa Contínua,Razão,Valor monetário total faturado pelo passageiro em pedidos de serviço de quarto (Room Service).,"Reais não-negativos em [0.0, 14,327.0] (Média: 224.69, Mediana: 0.0)."
8,FoodCourt,Numérico Real (float64),Quantitativa Contínua,Razão,Valor monetário total faturado pelo passageiro na praça de alimentação (Food Court).,"Reais não-negativos em [0.0, 29,813.0] (Média: 458.08, Mediana: 0.0)."
9,ShoppingMall,Numérico Real (float64),Quantitativa Contínua,Razão,Valor monetário total faturado pelo passageiro no centro comercial (Shopping Mall).,"Reais não-negativos em [0.0, 23,492.0] (Média: 173.73, Mediana: 0.0)."


---
## 4. Inspeção Detalhada por Tipologia Estatística

### 4.1. Variáveis Categóricas Nominais (`HomePlanet`, `Destination`, `CryoSleep`, `VIP`, `Transported`)
Distribuição de frequências absolutas e relativas (incluindo valores ausentes `NaN`) na base de treinamento.


In [6]:
colunas_nominais = ["HomePlanet", "Destination", "CryoSleep", "VIP", "Transported"]

resumo_nominais = []
for col in colunas_nominais:
    vc = df_train[col].value_counts(dropna=False)
    pct = df_train[col].value_counts(dropna=False, normalize=True) * 100
    for categoria, freq in vc.items():
        resumo_nominais.append({
            "Variável": col,
            "Categoria / Valor": "NaN (Ausente)" if pd.isna(categoria) else str(categoria),
            "Frequência Absoluta (n)": int(freq),
            "Frequência Relativa (%)": round(float(pct[categoria]), 2)
        })

df_resumo_nominais = pd.DataFrame(resumo_nominais)
display(df_resumo_nominais)


,Variável,Categoria / Valor,Frequência Absoluta (n),Frequência Relativa (%)
0,HomePlanet,Earth,4602,52.94
1,HomePlanet,Europa,2131,24.51
2,HomePlanet,Mars,1759,20.23
3,HomePlanet,NaN (Ausente),201,2.31
4,Destination,TRAPPIST-1e,5915,68.04
5,Destination,55 Cancri e,1800,20.71
6,Destination,PSO J318.5-22,796,9.16
7,Destination,NaN (Ausente),182,2.09
8,CryoSleep,False,5439,62.57
9,CryoSleep,True,3037,34.94


### 4.2. Variáveis Categóricas Estruturadas (`PassengerId`, `Cabin`, `Name`)
Verificação empírica dos padrões estruturais que embasarão a engenharia de variáveis derivadas (`KAN-12`).


In [7]:
# Decomposição exploratória sem alterar o dataframe original
name_parts = df_train["Name"].dropna().str.split(" ", n=1, expand=True)

resumo_estruturadas = pd.DataFrame([
    {
        "Variável": "PassengerId",
        "Padrão Estrutural": "gggg_pp (Grupo_Membro)",
        "Subcomponentes Extraíveis": "Group (gggg), MemberIndex (pp), GroupSize, IsAlone",
        "Estatísticas em train.csv": f"6.217 grupos únicos; tamanho do grupo varia de 1 a 8 passageiros."
    },
    {
        "Variável": "Cabin",
        "Padrão Estrutural": "deck/num/side",
        "Subcomponentes Extraíveis": "CabinDeck (A–G, T), CabinNum (0–1894), CabinSide (P=Bombordo, S=Boreste)",
        "Estatísticas em train.csv": f"8 decks {decks_unicos}; números de 0 a 1894; lados P ({int((cabin_parts[2]=='P').sum())}) e S ({int((cabin_parts[2]=='S').sum())})."
    },
    {
        "Variável": "Name",
        "Padrão Estrutural": "FirstName LastName",
        "Subcomponentes Extraíveis": "FirstName (Prenome), FamilyName / LastName (Sobrenome), FamilySize",
        "Estatísticas em train.csv": f"{name_parts[0].nunique()} prenomes únicos e {name_parts[1].nunique()} sobrenomes de família únicos."
    }
])

display(resumo_estruturadas)


,Variável,Padrão Estrutural,Subcomponentes Extraíveis,Estatísticas em train.csv
0,PassengerId,gggg_pp (Grupo_Membro),"Group (gggg), MemberIndex (pp), GroupSize, IsAlone",6.217 grupos únicos; tamanho do grupo varia de 1 a 8 passageiros.
1,Cabin,deck/num/side,"CabinDeck (A–G, T), CabinNum (0–1894), CabinSide (P=Bombordo, S=Boreste)","8 decks ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'T']; números de 0 a 1894; lados P (4206) e S (4288)."
2,Name,FirstName LastName,"FirstName (Prenome), FamilyName / LastName (Sobrenome), FamilySize",2706 prenomes únicos e 2217 sobrenomes de família únicos.


### 4.3. Variáveis Quantitativas Discreta (`Age`) e Contínuas (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`)
Estatísticas descritivas de posição, dispersão, quantis e proporção de zeros para as variáveis numéricas.


In [8]:
colunas_quantitativas = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

desc_quant = df_train[colunas_quantitativas].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T
desc_quant["Tipologia"] = [
    "Quantitativa Discreta" if c == "Age" else "Quantitativa Contínua"
    for c in desc_quant.index
]
desc_quant["Ausentes (NaN)"] = [int(df_train[c].isna().sum()) for c in desc_quant.index]
desc_quant["Zeros (== 0)"] = [int((df_train[c] == 0).sum()) for c in desc_quant.index]
desc_quant["% Zeros (entre válidos)"] = [
    round((df_train[c] == 0).sum() / df_train[c].notna().sum() * 100, 2)
    for c in desc_quant.index
]

cols_order = [
    "Tipologia", "count", "Ausentes (NaN)", "Zeros (== 0)", "% Zeros (entre válidos)",
    "mean", "std", "min", "25%", "50%", "75%", "90%", "95%", "99%", "max"
]
display(desc_quant[cols_order].round(2))


,Tipologia,count,Ausentes (NaN),Zeros (== 0),% Zeros (entre válidos),mean,std,min,25%,50%,75%,90%,95%,99%,max
Age,Quantitativa Discreta,8514.0,179,178,2.09,28.83,14.49,0.0,19.0,27.0,38.0,49.0,56.00,65.00,79.0
RoomService,Quantitativa Contínua,8512.0,181,5577,65.52,224.69,666.72,0.0,0.0,0.0,47.0,753.0,1274.25,3096.23,14327.0
FoodCourt,Quantitativa Contínua,8510.0,183,5456,64.11,458.08,1611.49,0.0,0.0,0.0,76.0,1026.0,2748.50,8033.31,29813.0
ShoppingMall,Quantitativa Contínua,8485.0,208,5587,65.85,173.73,604.70,0.0,0.0,0.0,27.0,619.6,927.80,2333.44,23492.0
Spa,Quantitativa Contínua,8510.0,183,5324,62.56,311.14,1136.71,0.0,0.0,0.0,59.0,732.0,1607.10,5390.10,22408.0
VRDeck,Quantitativa Contínua,8505.0,188,5495,64.61,304.85,1145.72,0.0,0.0,0.0,46.0,732.6,1534.20,5646.68,24133.0
